# The Minetti Model
*Energy cost of running on slopes: the polynomial, its predictions,
and its limits.*

Companion notebook for the blog article published on [gregs1t.github.io](https://gregs1t.github.io/trail/).

This notebook does not require a FIT file. It visualizes the
Minetti et al. (2002) polynomial and its properties.

## Installation

Run this cell once if needed.

In [ ]:
# Uncomment and run if needed
# !pip install numpy plotly

## Imports

In [ ]:
import os
import warnings

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
pio.renderers.default = "notebook"

print("Imports OK.")


---
## User parameters

Set `EXPORT_FOR_BLOG = True` to save figures as HTML.

In [ ]:
# --- Blog export ---
EXPORT_FOR_BLOG = True
EXPORT_DIR = os.path.join(
    "assets", "plotly", "2025-11_minetti_model"
)

if EXPORT_FOR_BLOG:
    os.makedirs(EXPORT_DIR, exist_ok=True)
    print(f"Export directory: {EXPORT_DIR}")


---
## 1. The Minetti polynomial

From Minetti et al. (2002), *J. Appl. Physiol.*, 93(3), 1039-1046.

$$C_r(i) = 155.4\,i^5 - 30.4\,i^4 - 43.3\,i^3 + 46.3\,i^2 + 19.5\,i + 3.6$$

where $i$ is the slope as a fraction (e.g. 0.10 for +10%).

In [ ]:
def minetti_cost(slope_pct):
    """Energy cost of running (J/kg/m) from Minetti et al. (2002)."""
    i = np.asarray(slope_pct, dtype=float) / 100.0
    return (
        155.4 * i**5
        - 30.4 * i**4
        - 43.3 * i**3
        + 46.3 * i**2
        + 19.5 * i
        + 3.6
    )


def minetti_cost_ratio(slope_pct):
    """Cost ratio relative to flat ground (Cr / Cr_flat)."""
    cr = minetti_cost(slope_pct)
    cr_flat = 3.6
    return np.clip(cr / cr_flat, 0.1, None)


## 2. Cost curve: absolute and relative

Two panels: absolute cost in J/kg/m (left) and multiplicative
factor vs flat (right).

In [ ]:
slopes = np.linspace(-45, 45, 500)
cr = minetti_cost(slopes)
ratio = minetti_cost_ratio(slopes)

fig_cost = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Energy cost of running (Minetti et al., 2002)",
        "Cost ratio vs flat ground"
    ),
    horizontal_spacing=0.12,
)

# Left panel: absolute cost
fig_cost.add_trace(go.Scatter(
    x=slopes, y=cr,
    mode="lines",
    line=dict(color="darkorange", width=2.5),
    name="Cr (J/kg/m)",
    hovertemplate=(
        "Slope: %{x:.0f}%<br>Cr: %{y:.2f} J/kg/m<extra></extra>"
    ),
), row=1, col=1)

fig_cost.add_hline(
    y=3.6, line_dash="dot", line_color="grey",
    annotation_text="flat: 3.6 J/kg/m",
    annotation_position="top right",
    row=1, col=1,
)

# Mark descent minimum
idx_min = np.argmin(cr[slopes < 0])
slope_min = slopes[slopes < 0][idx_min]
cr_min = cr[slopes < 0][idx_min]
fig_cost.add_trace(go.Scatter(
    x=[slope_min], y=[cr_min],
    mode="markers+text",
    marker=dict(size=8, color="steelblue"),
    text=[f"min: {cr_min:.2f} at {slope_min:.0f}%"],
    textposition="top right",
    textfont=dict(size=10),
    showlegend=False,
), row=1, col=1)

# Right panel: ratio
fig_cost.add_trace(go.Scatter(
    x=slopes, y=ratio,
    mode="lines",
    line=dict(color="steelblue", width=2.5),
    name="Cost ratio",
    hovertemplate=(
        "Slope: %{x:.0f}%<br>Ratio: %{y:.2f}x<extra></extra>"
    ),
), row=1, col=2)

fig_cost.add_hline(
    y=1.0, line_dash="dot", line_color="grey",
    annotation_text="flat = 1x",
    annotation_position="top right",
    row=1, col=2,
)

fig_cost.update_xaxes(title_text="Slope (%)", row=1, col=1)
fig_cost.update_xaxes(title_text="Slope (%)", row=1, col=2)
fig_cost.update_yaxes(title_text="Cr (J/kg/m)", row=1, col=1)
fig_cost.update_yaxes(title_text="Ratio vs flat", row=1, col=2)

fig_cost.update_layout(
    height=300,
    template="plotly_dark",
    showlegend=False,
    margin=dict(l=60, r=40, t=50, b=50),
)

if EXPORT_FOR_BLOG:
    fig_cost.write_html(
        os.path.join(EXPORT_DIR, "minetti_curve.html"),
        include_plotlyjs="cdn",
        full_html=True,
    )
    print("Exported: minetti_curve.html")

fig_cost.show()


---
## 3. Polynomial artifact at extreme slopes

Below roughly -55%, the polynomial produces negative energy costs.
Physically meaningless. Always clip or restrict the domain.

In [ ]:
slopes_ext = np.linspace(-70, 50, 600)
cr_ext = minetti_cost(slopes_ext)

fig_art = go.Figure()
fig_art.add_trace(go.Scatter(
    x=slopes_ext, y=cr_ext,
    mode="lines",
    line=dict(color="darkorange", width=2),
    hovertemplate=(
        "Slope: %{x:.0f}%<br>Cr: %{y:.2f}<extra></extra>"
    ),
))

fig_art.add_hline(y=0, line_dash="dash", line_color="red",
                  line_width=1.5)
fig_art.add_vrect(
    x0=-70, x1=-45,
    fillcolor="red", opacity=0.08,
    annotation_text="outside measured range",
    annotation_position="top left",
    annotation_font_size=10,
)
fig_art.add_vrect(x0=45, x1=50, fillcolor="red", opacity=0.08)

fig_art.update_layout(
    title="Minetti polynomial: artifact at extreme slopes",
    xaxis_title="Slope (%)",
    yaxis_title="Cr (J/kg/m)",
    template="plotly_dark",
    height=300,
    margin=dict(l=60, r=40, t=50, b=50),
    showlegend=False,
)

if EXPORT_FOR_BLOG:
    fig_art.write_html(
        os.path.join(EXPORT_DIR, "minetti_artifact.html"),
        include_plotlyjs="cdn",
        full_html=True,
    )
    print("Exported: minetti_artifact.html")

fig_art.show()


---
## 4. Practical reference table

Cost at standard gradients, and the implied GAP ratio.

In [ ]:
ref_slopes = [-20, -15, -10, -5, 0, 5, 10, 15, 20, 25, 30, 40]

print(f"{'Slope (%)':<12} {'Cr (J/kg/m)':<14} {'Ratio vs flat':<14}")
print("-" * 40)
for s in ref_slopes:
    cr_val = minetti_cost(s)
    ratio_val = minetti_cost_ratio(s)
    print(f"{s:>+8}%    {cr_val:>10.2f}     {ratio_val:>10.2f}x")


---
## 5. Minetti (running, 2002) vs Margaria (walking, 1938)

Margaria's seminal measurements covered a narrower range
and were limited to walking.

In [ ]:
# Margaria (1938) approximate walking cost data (Cw, J/kg/m)
margaria_slopes = np.array(
    [-25, -20, -15, -10, -5, 0, 5, 10, 15, 20, 25]
)
margaria_cost = np.array(
    [4.1, 2.4, 1.6, 1.3, 1.6, 2.5, 4.2, 5.9, 7.8, 9.8, 12.0]
)

slopes_plot = np.linspace(-45, 45, 500)

fig_comp = go.Figure()

fig_comp.add_trace(go.Scatter(
    x=slopes_plot, y=minetti_cost(slopes_plot),
    mode="lines",
    line=dict(color="darkorange", width=2.5),
    name="Minetti Cr (running, 2002)",
))

fig_comp.add_trace(go.Scatter(
    x=margaria_slopes, y=margaria_cost,
    mode="markers+lines",
    line=dict(color="mediumseagreen", width=1.5, dash="dash"),
    marker=dict(size=7, color="mediumseagreen"),
    name="Margaria Cw (walking, 1938)",
))

fig_comp.update_layout(
    title="Minetti (2002, running) vs Margaria (1938, walking)",
    xaxis_title="Slope (%)",
    yaxis_title="Cost of transport (J/kg/m)",
    template="plotly_dark",
    height=300,
    margin=dict(l=60, r=40, t=50, b=50),
    legend=dict(x=0.02, y=0.98),
)

if EXPORT_FOR_BLOG:
    fig_comp.write_html(
        os.path.join(EXPORT_DIR, "minetti_vs_margaria.html"),
        include_plotlyjs="cdn",
        full_html=True,
    )
    print("Exported: minetti_vs_margaria.html")

fig_comp.show()


---
## Output summary

In [ ]:
if EXPORT_FOR_BLOG:
    print("Generated files:")
    for f in sorted(os.listdir(EXPORT_DIR)):
        size = os.path.getsize(os.path.join(EXPORT_DIR, f))
        print(f"  {f:40s} {size / 1024:.1f} kB")


---
## What's next

Now that we understand the model behind GAP, we can implement
it on real data: computing Grade Adjusted Pace and VAM from
the slope column built in notebook 02.